In [1]:
# 0. Verisetini Hazırla
import pandas as pd
df = pd.read_csv("data/winequality_combined.csv")
df.head()

# Grover Dilemma: Type A'dan kaçınmak için 6500 satırlık verisetini kırpıyoruz.
# Sebep: log2(6500) yaklaşık= 13 Kübit. Olası indis durumları: 2^13 = 8192 !! 1692 SAHTE İNDİS DEMEK !!
# Arama sonucu olmayan indis sonuçlarını döneceği için 2^12 = 4098 GERÇEK İNDİS DEĞER'e eşledik böylece-
# Type A Grover Dilemma'sından kurtulmuş olduk.
df_ = df.copy().iloc[:4096]

alcohol_ideal_decimal = 2
density_ideal_decimal = 4
df_["alcohol"] = (df["alcohol"].round(alcohol_ideal_decimal) * 10**2).round().astype(int)
df_["density"] = (df["density"].round(density_ideal_decimal) * 10**4).round().astype(int)
print("Değişkenlerin en fazla alabildiği ondalık terim sayısı:")
for col in [col for col in df.columns if col not in ["alcohol", "density", "type"]]:
    max_decimal = df_[col].astype(str).str.split('.').str[1].fillna('').str.len().max()
    print("",col, max_decimal)
    df_[col] = (df[col].round(max_decimal) * (10 ** max_decimal)).round().astype(int)
## 'type' bağımlı değişken olduğu için 0 veya 1 değerleri atansa yeterli.
df_["type"] = (df_["type"]=="white").astype(int)
print("Veriler hazır. Veriseti artık QROM + SAT boru hattı için hazır.")
df_

Değişkenlerin en fazla alabildiği ondalık terim sayısı:
 fixed acidity 1
 volatile acidity 3
 citric acid 2
 residual sugar 2
 chlorides 3
 free sulfur dioxide 1
 total sulfur dioxide 1
 pH 2
 sulphates 2
 quality 0
Veriler hazır. Veriseti artık QROM + SAT boru hattı için hazır.


,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality,type
0,74,700,0,190,76,110,340,9978,351,56,940,5,0
1,78,880,0,260,98,250,670,9968,320,68,980,5,0
2,78,760,4,230,92,150,540,9970,326,65,980,5,0
3,112,280,56,190,75,170,600,9980,316,58,980,6,0
4,74,700,0,190,76,110,340,9978,351,56,940,5,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
4091,61,280,24,1995,74,320,1740,9992,319,44,930,6,1
4092,76,310,23,1270,54,200,1390,9984,316,50,970,4,1
4093,76,310,23,1270,54,200,1390,9984,316,50,970,4,1
4094,63,180,22,150,43,450,1550,9924,319,48,1020,5,1


In [ ]:
# 1. Python + QDK entegrasyonu gerçek bir Azure Workspace'i ile uygulanmaya hazır hale getiriliyor.

# Varsayılan olarak 'rigetti.sim.qvm' devre simülatörü seçilmiştir. Dilerseniz farklı bir simülasyon-
# veya fiziksel Kuantum devreleri seçebilirsiniz. 
# Not: Azure ortamında çalışmak için Azure'a kayıt olunmalı, 'Quantum Workspace' sorgusu sonrası ilgili-
# adımları uygulayarak Kuantum Çalışma Alanı oluşturulmalı ve dilediğiniz devre API'ını planınıza dahil-
# edip işlemleri tamamlamalısınız. Sonra ilgili Çalışma Alanına gidip 'Resource ID' parametresini kopya-
# layıp '.env' dosyası içerisindeki 'RESOURCE_ID' alanına yapıştırın.

from qdk import qsharp
from azure.quantum import Workspace
from dotenv import load_dotenv
import os

load_dotenv()
qsharp.init(project_root=".", target_profile=qsharp.TargetProfile.Base)
resource_id = os.getenv("RESOURCE_ID")
print("Azure Çalışma Ortamı Kaynak Kimliği:",resource_id + "\n")
workspace = Workspace(resource_id=resource_id)
targets = workspace.get_targets()
print("### Mevcut Çalıştırma Hedefleri ###")
for i, t in enumerate(targets, 1):
    print(f"{i}. Hedef:\n İsim: {t.name}\n Ort. Gecikme(s->saniye): {t.average_queue_time}\n Durum:{'MEVCUT' if t.current_availability == 'Available' else 'KULLANIM DIŞI'}")
target_str = "rigetti.sim.qvm"

Azure Çalışma Ortamı Kaynak Kimliği: /subscriptions/f5f81187-0841-4323-b7c9-26f533275f3d/resourceGroups/AzureQuantum/providers/Microsoft.Quantum/Workspaces/quantum-ws-56391312

### Mevcut Çalıştırma Hedefleri ###
1. Hedef:
 İsim: rigetti.sim.qvm
 Ort. Gecikme(s->saniye): 5
 Durum:MEVCUT
2. Hedef:
 İsim: quantinuum.sim.h2-1sc
 Ort. Gecikme(s->saniye): 1
 Durum:MEVCUT
3. Hedef:
 İsim: quantinuum.sim.h2-1e
 Ort. Gecikme(s->saniye): 99993
 Durum:MEVCUT

'rigetti.sim.qvm' seçiliyor...


In [ ]:
# 2. Grover Algoritması içerisine gönderilecek olan değişken ve aritmetik işlemler Q#'a uygun hale getirmek için ara işlemler uyguluyoruz.
OPS = {"==": 0, ">": 1, "<": 2, ">=": 3, "<=": 4, "!=":5}
kolon_sirasi = ["fixed acidity", "volatile acidity", "citric acid",
                "residual sugar", "chlorides", "free sulfur dioxide",
                "total sulfur dioxide", "pH", "sulphates",
                "alcohol", "density", "quality", "type"]


In [16]:
# 3. Her bir sütun için bit genişliği bilgilerini oluşturup saklamalıyız. Böylece Q#'ta belirli sorgu için yapılacak olan kübit işlemleri doğru şekilde
# uygulanmış olur

genislik = {}
print(f"Sütunların bit genişlikleri:")
for col in kolon_sirasi:
    max_val = int(df_[col].max())
    
    bit_length = max_val.bit_length()
    print(f" {col}: {bit_length}")
    genislik[col] = bit_length



def satir_to_bits(satir):
    tum_bitler = []

    for col in kolon_sirasi:
        deger = int(satir[col])
        
        bit_sayisi = genislik[col]

        ikili_string = format(deger, f"0{bit_sayisi}b")
        for karakter in ikili_string:
            if karakter == '1':
                tum_bitler.append(True)
            else:
                tum_bitler.append(False)

    return tum_bitler

dataset = df_[kolon_sirasi].apply(satir_to_bits, axis=1).tolist()

# Kontrol için ilk satırı yazdıralım:
print(dataset[0])
# çıktı: [False, False, True, True, ...] gibi uzun bir liste olacak

# Yapısal bilgiler
print(len(dataset))



Sütunların bit genişlikleri:
 fixed acidity: 8
 volatile acidity: 11
 citric acid: 8
 residual sugar: 12
 chlorides: 10
 free sulfur dioxide: 11
 total sulfur dioxide: 12
 pH: 9
 sulphates: 8
 alcohol: 11
 density: 14
 quality: 4
 type: 1
[False, True, False, False, True, False, True, False, False, True, False, True, False, True, True, True, True, False, False, False, False, False, False, False, False, False, False, False, False, False, False, True, False, True, True, True, True, True, False, False, False, False, True, False, False, True, True, False, False, False, False, False, False, True, True, False, True, True, True, False, False, False, False, True, False, True, False, True, False, True, False, False, True, False, True, False, True, True, True, True, True, False, False, True, True, True, False, False, False, False, True, True, True, False, True, False, True, True, False, False, True, False, False, True, True, False, True, True, True, True, True, False, True, False, False, True, F

In [ ]:
# 4. Q#'ın sütun değişkenlerini ayırt etmesi için bit-offset oluşturuyoruz.
offset = {}
konum = 0
print("Her bir sütunun bit-offset değeri:")
for col in kolon_sirasi:
    offset[col]=konum
    print(f" {col}: {konum}")
    konum+=genislik[col]

Her bir sütunun bit-offset değeri:
 fixed acidity: 0
 volatile acidity: 8
 citric acid: 19
 residual sugar: 27
 chlorides: 39
 free sulfur dioxide: 49
 total sulfur dioxide: 60
 pH: 72
 sulphates: 81
 alcohol: 89
 density: 100
 quality: 114
 type: 118


In [22]:
# 5. Q# içerisinde kullanmak için uyumlu bir betik sorgusu formatı oluşturuyoruz.
queries = [
    # (offset, genislik, operatör, değer)
    (offset["fixed acidity"], genislik["fixed acidity"], OPS["=="], 740),
    (offset["quality"],         genislik["quality"],         OPS[">="], 7),
]

In [ ]:
# 6. Q# devresine işlemler gönderilir ve hesaplama başlatılır.
# Dönecek olan sonuç
try:
    print(f"'{target_str}' seçiliyor...")
    target = workspace.get_targets(target_str)
except Exception as e:
    print(f"Bilinmeyen bir hata oluştu:\n{e}")
print("Seçim tamamlandı. Derleme işlemine geçildi.\n")
app_class = "Main.GroverSearchAlgorithm"
op = qsharp.eval(app_class)
print(f"'{app_class}' Sınıfı derleniyor...")
program = qsharp.compile(op, queries, dataset)
print(f"Derleme bitti. İş akışı {target_str}'e gönderildi.")
job = target.submit(program, "MicrosoftFY26GroverJob", shots=100)
print("İş akışı tamamlandı. Sonuç:")
results = job.get_results()
print(results)